# ema-first-moment — worked example 2: First-moment EMA across a parameter list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-first-moment`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A real optimizer holds one first-moment buffer per parameter. The update walks `zip(m_list, grad_list)` and applies `m = beta1*m + (1-beta1)*g` in place to each, returning the (same) buffers by reference. Each buffer keeps its own storage so optimizer state never detaches from its parameter.

## Worked solution

We apply the first-moment update to a list of buffers, one gradient each.

1. We have `m_list` (the per-parameter first-moment buffers, to be mutated) and `grad_list` (gradients, left untouched), the same length.
2. We iterate `zip(m_list, grad_list)`. For each pair we compute `beta1*m + (1-beta1)*g` and write it back with `m.copy_(...)`.
3. We append `m` (the same object, by reference) to the output so callers can chain, but the mutation already happened in place.
4. After the loop, each buffer holds the one-step EMA of its gradient. We verify one entry by hand: starting from zero, one step gives `(1-beta1)*g` exactly.

In [ ]:
import torch as t

t.manual_seed(1)
beta1 = 0.9
m_list = [t.zeros(2), t.zeros(3)]
grad_list = [t.tensor([2.0, -4.0]), t.tensor([1.0, 1.0, -3.0])]

def ema_m_step_list(m_list, grad_list, beta1):
    out = []
    for m, g in zip(m_list, grad_list):
        m.copy_(beta1 * m + (1 - beta1) * g)
        out.append(m)
    return out

out = ema_m_step_list(m_list, grad_list, beta1)
print([o.tolist() for o in out])
print('first step is (1-b1)*g:', bool(t.allclose(out[0], (1 - beta1) * grad_list[0])))